# Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. summarizaation iss usefull for the following :
- Long running conversation that exceeds context window
- multi turn dialogues with extensive history
- applications where preserving full convesation context matters

In [2]:
import os
from dotenv import find_dotenv,load_dotenv
load_dotenv(find_dotenv())
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware 
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

### Message size


In [4]:
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger=("messages",10),
            keep=("messages", 4)
        )
    ]
)

In [5]:
# run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [6]:
# Alternatively test data

questions = [
    "what is 3+4?",
    "what is 9*9?",
    "what is 19/19?",
    "what is 30+89?",
    "what is 3-90?",
    "what is 97-87?",
    ]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages : {response}")
    print(f"Messages : {len(response["messages"])}")


Messages : {'messages': [HumanMessage(content='what is 3+4?', additional_kwargs={}, response_metadata={}, id='d79b3d00-4fd6-4346-aaec-4024b3067348'), AIMessage(content='3 + 4 = 7.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.010770833, 'completion_tokens_details': None, 'prompt_time': 0.002735895, 'prompt_tokens_details': None, 'queue_time': 0.047878864, 'total_time': 0.013506728}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7eb4-2b68-7540-9a46-73c1a94bb784-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages : 2
Messages : {'messages': [HumanMessage(content='what is 3+4?', additional_kwargs={}, response_metadata={}, id='d79b3d00-4fd6-4346-aaec-4024b3067348'), AIMess

### token size

In [7]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [8]:
@tool
def Search_hostels(city:str)-> str:
    """Always use this tool to search for hotels, hostels, or lodging in a specific city."""
    return f"hoetel in the {city}:  1.(greand hotel - 5 star ,$200/night , spa ,pool) 2.(City inn - 3 star , $50 /night , pool, gym ) 3.(budget city ,$29/night , near main market and bussness center)"

agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools = [Search_hostels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = "groq:llama-3.1-8b-instant",
            trigger=("tokens", 550),
            keep = ("tokens", 200)
        )
    ]
)


def Count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars


In [9]:
# run test 
cities = [ "London","tokyo" ,"New York" ,"Dubai" ,"Delhi"]

    
for city in cities:
    # Create a NEW config with a unique thread_id for each city
    config = {"configurable": {"thread_id": f"test-{city}"}}
    response = agent.invoke({"messages":[HumanMessage(content=f"find hotel in {city}")]}, config)

    tokens = Count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response["messages"])} messages")
    print(f"{(response["messages"])}")
    

London: ~730 tokens, 6 messages
[HumanMessage(content='find hotel in London', additional_kwargs={}, response_metadata={}, id='12df17be-7f10-4dc0-929c-ff3c6d76fc2f'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qkvx72k5k', 'function': {'arguments': '{"city":"London"}', 'name': 'Search_hostels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 242, 'total_tokens': 258, 'completion_time': 0.035018816, 'completion_tokens_details': None, 'prompt_time': 0.026999064, 'prompt_tokens_details': None, 'queue_time': 0.057370315, 'total_time': 0.06201788}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7eb4-3a4e-7de1-9e98-891f6ec9a7e1-0', tool_calls=[{'name': 'Search_hostels', 'args': {'city': 'London'}, 'id': 'qkvx72k5k', 'type': 'tool_call'}], invalid_tool_calls=[], usage_me

### Fraction


In [10]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage 
from langchain.agents.middleware import summarization
from langgraph.checkpoint.memory import InMemorySaver



@tool
def Search_hostels(city:str)-> str:
    """ Search hostel - returns long response to use more tokens. """
    return f"hoetel in the {city}:  1.(greand hotel - 5 star ,$200/night , spa ,pool) 2.(City inn - 3 star , $50 /night , pool, gym ) 3.(budget city ,$29/night , near main market and bussness center)"

agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools = [Search_hostels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model = "groq:llama-3.1-8b-instant",
            trigger=("fraction" , 0.005),
            keep = ("fraction" , 0.002)
        )
    ]
)
config= {"configurable":{"thread_id":"test-1"}}

def Count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars

# run test 
cities = [ "London","tokyo" ,"New York" ,"Dubai" ,"Delhi"]

for city in cities:
    response = agent.invoke({"messages":[HumanMessage(content=f"find hotel in {city}")]}, config)

    tokens = Count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response["messages"])} messages")
    print(f"{(response["messages"])}")

London: ~1080 tokens, 6 messages
[HumanMessage(content='find hotel in London', additional_kwargs={}, response_metadata={}, id='0692c41b-9864-49ec-b8a7-d947952d2a4b'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'en462d9w3', 'function': {'arguments': '{"city":"London"}', 'name': 'Search_hostels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 226, 'total_tokens': 242, 'completion_time': 0.047494478, 'completion_tokens_details': None, 'prompt_time': 0.023665744, 'prompt_tokens_details': None, 'queue_time': 0.056778485, 'total_time': 0.071160222}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7eb4-7864-7f73-98a9-aab6f2e45a19-0', tool_calls=[{'name': 'Search_hostels', 'args': {'city': 'London'}, 'id': 'en462d9w3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [11]:
from langchain.agents import  create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def Read_mail_tool(email_id:str)-> str:
    """ mock function to read an email by  its ID """
    return f"Email content for ID {email_id}"

def Send_email_tool(recipient :str, subject:str, body:str ) ->str:
    """ mock function to send email """
    return f"Email send to {recipient} with subject {subject}"




In [12]:
agent = create_agent(
    model = "groq:llama-3.1-8b-instant",
    tools=[Read_mail_tool, Send_email_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "Send_email_tool":{
                    "allowed decision":["approve" , "reject" ,"edit"]
                },
                "Read_mail_tool":False,
            }
        )
    ]
)

In [15]:
config ={"configurable":{"thread_id":"test-approve"}}

#step 1 result

result = agent.invoke({"messages":[HumanMessage(content="Send email to 'jhondeo@gamail.com'  with sunject 'Hello' and body 'how are you?' ")]},
config=config
)



In [16]:
result

{'messages': [HumanMessage(content="Send email to 'jhondeo@gamail.com'  with sunject 'Hello' and body 'how are you?' ", additional_kwargs={}, response_metadata={}, id='2abf2c30-94bd-4165-88c9-2212ebfab70c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fzxwd01ev', 'function': {'arguments': '{"body":"how are you?","recipient":"jhondeo@gamail.com","subject":"Hello"}', 'name': 'Send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 318, 'total_tokens': 354, 'completion_time': 0.038768343, 'completion_tokens_details': None, 'prompt_time': 0.022181343, 'prompt_tokens_details': None, 'queue_time': 0.049101253, 'total_time': 0.060949686}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7eca-a16c-7462-99e5-5ec41d1fea60-0', tool_calls=[{'name': 'Send_email_tool', 

In [21]:
#step 2 result
from langgraph.types import Command

if "__interrupt__" in result:
    print(" Paused! Approving" )

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config 
    )

    print(f" Result: {result['message'][-1].content}")

In [18]:
result

{'messages': [HumanMessage(content="Send email to 'jhondeo@gamail.com'  with sunject 'Hello' and body 'how are you?' ", additional_kwargs={}, response_metadata={}, id='2abf2c30-94bd-4165-88c9-2212ebfab70c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fzxwd01ev', 'function': {'arguments': '{"body":"how are you?","recipient":"jhondeo@gamail.com","subject":"Hello"}', 'name': 'Send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 318, 'total_tokens': 354, 'completion_time': 0.038768343, 'completion_tokens_details': None, 'prompt_time': 0.022181343, 'prompt_tokens_details': None, 'queue_time': 0.049101253, 'total_time': 0.060949686}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7eca-a16c-7462-99e5-5ec41d1fea60-0', tool_calls=[{'name': 'Send_email_tool', 